# Nepal flood modelling: split, feature engineering, feature freeze

In [23]:
import json, re, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "processed" / "nepal_flood_clean.csv").exists())
OUT_DIR = ROOT / "data" / "frozen"
OUT_DIR.mkdir(exist_ok=True)


TRAIN_END  = "2025-12-31"
TEST_START = "2026-01-01"

CV_FOLDS = [
    {"name": "fold1", "train_end": "2023-12-31", "val_start": "2024-01-01", "val_end": "2024-12-31"},
    {"name": "fold2", "train_end": "2024-12-31", "val_start": "2025-01-01", "val_end": "2025-12-31"},]
#For Feature
TARGET_RAW = "river_discharge_m3s"
RAIN_WINDOWS = [3, 7]            # days, window ends on day t (includes today)
DISCHARGE_LAGS = [1, 2, 3, 7]    # days back, strictly before day t
MAX_WINDOW = 7                   # longest look-back, rows lost at the start of each station

## 1. Load the Data
since there are some days with no discharge, hence it uses: $$\text{log1p}(x) = \ln(1 + x)$$

In [24]:
df = pd.read_csv(ROOT / "data" / "processed" / "nepal_flood_clean.csv")
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")
df = df.sort_values(["location", "date"]).reset_index(drop=True)

assert not df.duplicated(["location", "date"]).any(), "duplicate station-days"
gaps = df.groupby("location")["date"].apply(lambda s: int((s.diff().dt.days.dropna() != 1).sum()))
assert (gaps == 0).all(), "missing days in a station series"
assert df[TARGET_RAW].ge(0).all()

STATIONS = sorted(df["location"].unique())
STATION_KEYS = [re.sub(r"\W+", "_", s).strip("_") for s in STATIONS]

print("Rows:", len(df), "| Stations:", len(STATIONS))
print("Dates:", df["date"].min().date(), "to", df["date"].max().date())
print("Days with zero discharge:", int((df[TARGET_RAW] == 0).sum()), "target uses log1p")

Rows: 13390 | Stations: 10
Dates: 2023-01-01 to 2026-08-31
Days with zero discharge: 223 target uses log1p


## Hann’s Tableau dashboards — descriptive overview

The following dashboards were prepared by **Hann in Tableau** from the cleaned Nepal flood/weather dataset and supplied as screenshots. They are included here as visual documentation; the images are static, so their Tableau filters are not interactive.

**Scope:** these dashboards display the full available period, including 2026. They provide a retrospective description of the data, not independent evidence of predictive performance. Feature selection and model tuning must follow the training/validation boundaries defined below.


### Figure 1. Nepal station locations

![Map of Nepal showing the ten station locations represented in the dataset.](images/hann_nepal_station_map.png)

The supplied map shows the geographic spread of the dataset’s ten stations across Nepal. 



### Figure 2. River-discharge overview

![Hann’s Tableau dashboard showing daily river discharge, distribution summaries, quarterly discharge heatmap and log-discharge histogram.](images/hann_discharge_dashboard.png)

The time series and heatmap show recurring seasonal peaks and all the major differences between locations, Kusum stands out among all in the displayed discharge scale. The histogram shows a skewed distribution on the displayed logarithmic scale. These plots motivate inspecting station-specific behaviour and extreme values; they do not establish flood occurrence.



### Figure 3. Weather patterns and rain–discharge relationships

![Hann’s Tableau dashboard showing station-level precipitation and soil moisture, monthly rain–discharge relationships, and monthly rainfall patterns.](images/hann_weather_dashboard.png)

Monthly rainfall shows recurring seasonal peaks. The rain discharge panel suggests that the relationship varies across locations these combined associations do not establish causation or forecast skill.



## 2. Data Split

Train (2023-01-01 to 2025-12-31) = Three monsoon seasons. Cross-validation and the final model fit.  
Test (2026-01-01 to 2026-08-31) = Newest data, that scored once at the very end.

In [25]:

def assign_split(dates):
    return np.select([dates <= TRAIN_END, dates >= TEST_START], ["train", "test"], default="")

df["split"] = assign_split(df["date"])
assert (df["split"] != "").all()

summary = (
    df.assign(monsoon=df["date"].dt.month.isin([6, 7, 8, 9]))
      .groupby("split")
      .agg(start=("date", "min"), end=("date", "max"),
           rows=("date", "size"),
           monsoon_rows=("monsoon", "sum"),
           mean_discharge=(TARGET_RAW, "mean"))
      .loc[["train", "test"]]
)
summary["days_per_station"] = summary["rows"] // len(STATIONS)
display(summary.round({"mean_discharge": 1}))

,start,end,rows,monsoon_rows,mean_discharge,days_per_station
split,,,,,,
train,2023-01-01,2025-12-31,10960,3660,21.4,1096
test,2026-01-01,2026-08-31,2430,920,15.7,243


This is a split of the 13,390 row dataset into an 81.9% training set (2023–2025) and an 18.1% out-of-sample test set (2026), ensuring that all 10 river stations are evaluated on identical, contiguous time horizons without look-ahead leakage.

### Cross-validation inside the training set
Expanding-window folds, **split by date** (every station is cut at the same date):

| Fold | Train | Validate |
|---|---|---|
| fold1 | 2023 | 2024 |
| fold2 | 2023-2024 | 2025 |

However, the 1st fold trains on a single monsoon season and validates on 2024, the year with the largest flood (Kusum 3,176 m3/s vs 1,802 in 2023), so it is a pessimistic check.

In [26]:
def cv_splits(frame):
    """Yield (fold_name, train_idx, val_idx) as positional indices, split by date."""
    dates = frame["date"]
    for f in CV_FOLDS:
        tr = np.flatnonzero((dates <= f["train_end"]).to_numpy())
        va = np.flatnonzero(((dates >= f["val_start"]) & (dates <= f["val_end"])).to_numpy())
        assert dates.iloc[tr].max() < dates.iloc[va].min(), "train must end before validation starts"
        yield f["name"], tr, va


train_frame = df[df["split"] == "train"].reset_index(drop=True)
fold_rows = []
for name, tr, va in cv_splits(train_frame):
    v = train_frame.iloc[va]
    fold_rows.append({"fold": name, "train_rows": len(tr), "val_rows": len(va),
                      "val_monsoon_rows": int(v["date"].dt.month.isin([6, 7, 8, 9]).sum()),
                      "val_max_discharge": round(float(v[TARGET_RAW].max()), 0),
                      "train_max_discharge": round(float(train_frame.iloc[tr][TARGET_RAW].max()), 0)})
display(pd.DataFrame(fold_rows))

,fold,train_rows,val_rows,val_monsoon_rows,val_max_discharge,train_max_discharge
0,fold1,3650,3660,1220,3176.0,1802.0
1,fold2,7310,3650,1220,1190.0,3176.0


splitting the data into two sequential date-aligned folds across all 10 stations, Fold 1 trains on 2023 data and validates on 2024, serving as a pessimistic benchmark due to 2024's record-high flood peaks ($3,176 \text{ m}^3/\text{s}$ vs. 2023's $1,802 \text{ m}^3/\text{s}$). Fold 2 expands the training memory to cover 2023–2024 and validates on 2025. Hence, Both folds validate over an identical 1,220 monsoon station-days, which has consistent seasonal balance during cross-validation.

## 3. Investigate candidate features (training set only)

Two views are used because they answer different questions:
- correlation with the **level** of log discharge (heavily influenced by the season)
- correlation with the **day-to-day change** in log discharge (the signal that rain adds on top of what the river is already doing)

In [27]:
train_raw = df[df["split"] == "train"].copy()
train_raw["y"] = np.log1p(train_raw[TARGET_RAW])
train_raw["dy"] = train_raw["y"] - train_raw.groupby("location")["y"].shift(1)

WINDOWS = [1, 3, 7,]
for w in WINDOWS:
    train_raw[f"p{w}"] = train_raw.groupby("location")["precipitation_mm"].transform(
        lambda s, w=w: s.rolling(w, min_periods=w).sum())

def mean_station_corr(frame, a, b, method="spearman"):
    """Correlation computed inside each station, then averaged."""
    vals = []
    for _, d in frame.groupby("location"):
        d = d[[a, b]].dropna()
        vals.append(d[a].corr(d[b], method=method))
    return float(np.nanmean(vals))

rain_tbl = pd.DataFrame({
    "window_days": WINDOWS,
    "corr_with_level": [mean_station_corr(train_raw, "y", f"p{w}") for w in WINDOWS],
    "corr_with_daily_change": [mean_station_corr(train_raw, "dy", f"p{w}") for w in WINDOWS],
}).round(3)
display(rain_tbl)

,window_days,corr_with_level,corr_with_daily_change
0,1,0.462,0.069
1,3,0.497,0.087
2,7,0.539,0.004


The level correlation keeps rising with window length, but that mostly reflects the monsoon as wet months have both more rain and more flow. The daily-change correlation is positive for short windows (strongest at 2 to 3 days), fades to about zero at 7 days, and turns negative beyond that, because long windows describe the recession phase rather than new inflow. That supports using **3-day** (event signal) and **7-day** (short-term wetness) windows.

In [33]:
lag_rows = []
for k in [1, 2, 3, 7,]:
    vals = []
    for _, d in train_raw.groupby("location"):
        vals.append(d["y"].corr(d["y"].shift(k)))
    lag_rows.append({"lag_days": k, "mean_autocorr_log_q": round(float(np.nanmean(vals)), 3)})
display(pd.DataFrame(lag_rows))

weather = ["precipitation_mm", "precipitation_hours", "soil_moisture_0_100cm_m3m3",
           "temperature_mean_c", "dew_point_mean_c", "relative_humidity_mean_pct",
           "wind_speed_max_kmh", "wind_gusts_max_kmh"]

#spearman's correlation matrix
cm = train_raw[weather].corr(method="spearman").abs()

pairs = [(weather[i], weather[j], round(float(cm.iloc[i, j]), 3))
         for i in range(len(weather)) for j in range(i + 1, len(weather)) if cm.iloc[i, j] > 0.9]
print("Pairs with Spearman > 0.9:", pairs)

snow = (df["precipitation_mm"] - df["rain_mm"])
print("\nprecipitation > rain =", int((snow > 1e-6).sum()),", by station:", df.loc[snow > 1e-6, "location"].value_counts().to_dict())

,lag_days,mean_autocorr_log_q
0,1,0.981
1,2,0.960
2,3,0.946
3,7,0.904


Pairs with Spearman > 0.9: [('precipitation_mm', 'precipitation_hours', 0.979)]

precipitation > rain = 227 , by station: {'Rasuwagadhi': 223, 'Chameliya/Nayalbadi': 4}


As Yesterday's or tomorrow's discharge is extremely informative (autocorrelation close to 1), so any model with previous-discharge features must be judged against a **persistence baseline** (predict today = yesterday). Two feature sets are therefore frozen: `with_discharge` and `rain_only`.
- `precipitation_hours` is almost a duplicate of `precipitation_mm`, so it was is dropped.
- `rain_mm` was dropped because it differs from `precipitation_mm` only through snowfall, which is almost entirely Rasuwagadhi (winter and spring).
- `wind_direction_dominant_deg` was dropped cuz no linkage. 

## 4. Feature engineering
Target: `y = log1p(river_discharge_m3s)`. Back-transform with `np.expm1`.

In [36]:
FEATURE_GROUPS = {
    "weather_same_day": ["precipitation_mm", "soil_moisture_0_100cm_m3m3", "temperature_mean_c",
                         "dew_point_mean_c", "relative_humidity_mean_pct",
                         "wind_speed_max_kmh", "wind_gusts_max_kmh"],
    "rain_history": [f"precip_sum_{w}d" for w in RAIN_WINDOWS],
    "seasonal": ["doy_sin", "doy_cos", "is_monsoon"],
    "station": [f"station_{k}" for k in STATION_KEYS],
    "discharge_history": [f"log_q_lag{k}" for k in DISCHARGE_LAGS] + ["log_q_mean7"],
}
FEATURE_SETS = {
    "with_discharge": [c for g in FEATURE_GROUPS.values() for c in g],
    "rain_only": [c for k, g in FEATURE_GROUPS.items() if k != "discharge_history" for c in g],
}
ALL_FEATURES = FEATURE_SETS["with_discharge"]


def build_features(raw):
    """Return one row per station-day with the target and all engineered features."""
    d = raw.sort_values(["location", "date"]).reset_index(drop=True).copy()
    d["y"] = np.log1p(d[TARGET_RAW])
    g = d.groupby("location")

    # rainfall history: window ends on day t
    for w in RAIN_WINDOWS:
        d[f"precip_sum_{w}d"] = g["precipitation_mm"].transform(
            lambda s, w=w: s.rolling(w, min_periods=w).sum())

    # discharge history: strictly before day t
    for k in DISCHARGE_LAGS:
        d[f"log_q_lag{k}"] = g["y"].shift(k)
    d["log_q_mean7"] = g["y"].transform(lambda s: s.shift(1).rolling(7, min_periods=7).mean())

    # seasonality
    doy = d["date"].dt.dayofyear
    d["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    d["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    d["is_monsoon"] = d["date"].dt.month.isin([6, 7, 8, 9]).astype(int)

    # station indicator columns (station list is fixed, not learned from a split)
    key = d["location"].map(lambda s: re.sub(r"\W+", "_", s).strip("_"))
    for k in STATION_KEYS:
        d[f"station_{k}"] = (key == k).astype(int)
    return d


feat = build_features(df)
feat["split"] = assign_split(feat["date"])

# Drop the warm-up rows (first MAX_WINDOW days of each station) that lack full history
warmup = feat.groupby("location").cumcount() < MAX_WINDOW
print("Warm-up rows dropped:", int(warmup.sum()), "(all in the training period)")
feat_model = feat[~warmup].copy()
assert feat_model[ALL_FEATURES + ["y"]].notna().all().all()

display(feat_model.groupby("split").size().loc[["train", "test"]].rename("rows").to_frame())

Warm-up rows dropped: 70 (all in the training period)


,rows
split,
train,10890
test,2430


- Applied a log transformation to river discharge.
- Assigned observations to their respective data splits and removes the initial rows that lack sufficient historical data.
- Checked missing values and counts the remaining observations.
- These features are then prepared for feature selection and freezing.

## 5. Baselines to beat (cross-validation folds)
Creates two simple baselines, each scored on every fold's validation year using only that fold's training rows:
- **Persistence**: today's log discharge = yesterday's. The bar for models that use discharge history (`with_discharge`).
- **Climatology**: the station's mean log discharge for that calendar month. The bar for `rain_only`.


In [37]:
def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))
def r2(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float(1 - np.sum((a - b) ** 2) / np.sum((a - a.mean()) ** 2))

train_df = feat_model[feat_model["split"] == "train"].reset_index(drop=True)

rows, station_rows = [], []
for name, tr, va in cv_splits(train_df):
    fit, hold = train_df.iloc[tr], train_df.iloc[va]

    # climatology learned from this fold's training rows only
    clim = (fit.assign(month=fit["date"].dt.month)
               .groupby(["location", "month"])["y"].mean().rename("clim"))
    hold = hold.assign(month=hold["date"].dt.month).join(clim, on=["location", "month"])
    assert hold["clim"].notna().all()

    masks = {"all days": np.ones(len(hold), dtype=bool),
             "monsoon (Jun-Sep)": (hold["is_monsoon"] == 1).to_numpy(),
             "non-monsoon": (hold["is_monsoon"] == 0).to_numpy()}
    for subset, m in masks.items():
        h = hold[m]
        rows.append({"fold": name, "subset": subset,
                     "persistence_rmse": rmse(h["y"], h["log_q_lag1"]),
                     "persistence_r2": r2(h["y"], h["log_q_lag1"]),
                     "climatology_rmse": rmse(h["y"], h["clim"]),
                     "climatology_r2": r2(h["y"], h["clim"])})
    for loc, h in hold.groupby("location"):
        station_rows.append({"fold": name, "location": loc,
                             "persistence_rmse": rmse(h["y"], h["log_q_lag1"])})

baseline = pd.DataFrame(rows).round(3)
display(baseline)
display(pd.DataFrame(station_rows).pivot(index="location", columns="fold",
                                         values="persistence_rmse").round(3))

,fold,subset,persistence_rmse,persistence_r2,climatology_rmse,climatology_r2
0,fold1,all days,0.158,0.988,0.398,0.927
1,fold1,monsoon (Jun-Sep),0.259,0.976,0.559,0.888
2,fold1,non-monsoon,0.062,0.997,0.285,0.944
3,fold2,all days,0.148,0.990,0.407,0.926
4,fold2,monsoon (Jun-Sep),0.191,0.987,0.566,0.886
5,fold2,non-monsoon,0.120,0.991,0.296,0.943


fold,fold1,fold2
location,,
Bahrabise,0.109,0.083
Belsot,0.089,0.084
Bhada Bridge,0.192,0.080
Chameliya/Nayalbadi,0.051,0.038
Chatara,0.183,0.158
Chisapani,0.105,0.067
Devghat,0.068,0.049
Khokana,0.295,0.328
Kusum,0.227,0.230


## 6. Freeze
Writes the train and test datasets, the CV helper module, the feature configuration (JSON), and a human-readable feature dictionary. File hashes are stored so any later change to the frozen data is detectable.

In [38]:
DESCRIPTIONS = {
    "precipitation_mm": "Total precipitation on day t (mm).",
    "soil_moisture_0_100cm_m3m3": "Soil moisture, 0-100 cm layer, day t.",
    "temperature_mean_c": "Mean air temperature, day t (deg C).",
    "dew_point_mean_c": "Mean dew point, day t (deg C).",
    "relative_humidity_mean_pct": "Mean relative humidity, day t (%).",
    "wind_speed_max_kmh": "Maximum wind speed, day t (km/h).",
    "wind_gusts_max_kmh": "Maximum wind gust, day t (km/h).",
    "precip_sum_3d": "Sum of precipitation over days t-2..t (mm), per station.",
    "precip_sum_7d": "Sum of precipitation over days t-6..t (mm), per station.",
    "doy_sin": "sin(2*pi*day_of_year/365.25).",
    "doy_cos": "cos(2*pi*day_of_year/365.25).",
    "is_monsoon": "1 if month is June to September, else 0.",
    "log_q_lag1": "log1p(discharge) on day t-1.",
    "log_q_lag2": "log1p(discharge) on day t-2.",
    "log_q_lag3": "log1p(discharge) on day t-3.",
    "log_q_lag7": "log1p(discharge) on day t-7.",
    "log_q_mean7": "Mean of log1p(discharge) over days t-7..t-1.",
}
for s, k in zip(STATIONS, STATION_KEYS):
    DESCRIPTIONS[f"station_{k}"] = f"1 if station is {s}, else 0."

ID_COLS = ["date", "location"]
cols_out = ID_COLS + [TARGET_RAW, "y"] + ALL_FEATURES

(OUT_DIR / "val.csv").unlink(missing_ok=True)   # remove a stale file from an earlier version

# CV helper module so the modelling notebook uses exactly the same folds
CV_TEMPLATE = """import numpy as np

CV_FOLDS = __FOLDS__


def cv_splits(frame):
    # Yield (fold_name, train_idx, val_idx) as positional indices, split by date.
    dates = frame["date"]
    for f in CV_FOLDS:
        tr = np.flatnonzero((dates <= f["train_end"]).to_numpy())
        va = np.flatnonzero(((dates >= f["val_start"]) & (dates <= f["val_end"])).to_numpy())
        assert dates.iloc[tr].max() < dates.iloc[va].min()
        yield f["name"], tr, va


def cv_index_pairs(frame):
    # List of (train_idx, val_idx) pairs, usable as cv= in scikit-learn.
    return [(tr, va) for _, tr, va in cv_splits(frame)]
"""
CV_MODULE = CV_TEMPLATE.replace("__FOLDS__", json.dumps(CV_FOLDS, indent=4))
(OUT_DIR / "cv_folds.py").write_text(CV_MODULE)

hashes = {"cv_folds.py": hashlib.sha256((OUT_DIR / "cv_folds.py").read_bytes()).hexdigest()}
for name in ["train", "test"]:
    part = feat_model.loc[feat_model["split"] == name, cols_out]
    path = OUT_DIR / f"{name}.csv"
    part.to_csv(path, index=False)
    hashes[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()

config = {
    "target": {"column": "y", "definition": "log1p(river_discharge_m3s)",
               "inverse": "expm1", "raw_column_kept_for_evaluation": TARGET_RAW},
    "split": {"train": ["2023-01-08", TRAIN_END],
              "test": [TEST_START, str(df["date"].max().date())],
              "note": "Chronological, same dates for all stations. First 7 days per station dropped (warm-up)."},
    "cv": {"scheme": "expanding window, split by date (not by row)", "folds": CV_FOLDS,
           "helper": "data/frozen/cv_folds.py: cv_splits(frame), cv_index_pairs(frame)"},
    "id_columns": ID_COLS,
    "feature_groups": FEATURE_GROUPS,
    "feature_sets": FEATURE_SETS,
    "windows": {"rain_days": RAIN_WINDOWS, "discharge_lags": DISCHARGE_LAGS, "discharge_mean_days": 7},
    "dropped_columns": {
        "rain_mm": "Differs from precipitation_mm only via snowfall (mostly Rasuwagadhi).",
        "precipitation_hours": "Spearman ~0.98 with precipitation_mm.",
        "wind_direction_dominant_deg": "Circular variable, not examined; excluded.",
        "river, basin, dhm_station, latitude, longitude, elevation_m": "Station identifiers; covered by station indicator columns.",
    },
    "scaling": "None applied. If a model needs it (linear, neural), fit the scaler on train only.",
    "timing": ("Features at day t use weather observed on day t and discharge up to t-1, "
               "so the task is a one-day-ahead prediction. Day-t rainfall must be available "
               "(observed or forecast) when used operationally."),
    "descriptions": DESCRIPTIONS,
    "file_sha256": hashes,
}
(OUT_DIR / "feature_config.json").write_text(json.dumps(config, indent=2))

# Human-readable dictionary
lines = ["# Feature dictionary", "",
         f"Target: `y` = log1p({TARGET_RAW})", "",
         "Feature sets: `with_discharge` (all groups) and `rain_only` (no discharge_history).", ""]
for grp, cols in FEATURE_GROUPS.items():
    lines += [f"## {grp}", ""] + [f"- `{c}`: {DESCRIPTIONS[c]}" for c in cols] + [""]
(OUT_DIR / "FEATURES.md").write_text("\n".join(lines))

# Reload check
chk = {n: pd.read_csv(OUT_DIR / f"{n}.csv", parse_dates=["date"]) for n in ["train", "test"]}
assert chk["train"]["date"].max() < chk["test"]["date"].min()
assert all(list(c.columns) == cols_out for c in chk.values())
assert all(c[ALL_FEATURES].notna().all().all() for c in chk.values())

print("Frozen files:", sorted(p.name for p in OUT_DIR.iterdir()))
print({k: v.shape for k, v in chk.items()})
print("with_discharge features:", len(FEATURE_SETS["with_discharge"]),
      "| rain_only features:", len(FEATURE_SETS["rain_only"]))

# The frozen CV helper reproduces the folds on the saved training file
import sys
sys.path.insert(0, str(OUT_DIR))
from cv_folds import cv_index_pairs
pairs = cv_index_pairs(chk["train"])
print("CV folds (train rows, validation rows):", [(len(a), len(b)) for a, b in pairs])

Frozen files: ['FEATURES.md', 'cv_folds.py', 'feature_config.json', 'test.csv', 'train.csv']
{'train': (10890, 31), 'test': (2430, 31)}
with_discharge features: 27 | rain_only features: 22
CV folds (train rows, validation rows): [(3580, 3660), (7240, 3650)]


## 7.  neural-network section. The supplied training curves

### Figure 4. MLP training and validation loss

![Louis’s MLP training and validation binary cross-entropy loss over epochs.](images/louis_mlp_loss.png)

MLP loss-curve 

Both losses fall rapidly during the early epochs and then flatten, with validation loss remaining above training loss. The displayed validation curve does not show a sustained upward trend, but this figure alone cannot establish that overfitting is absent or quantify performance on the test set. Where validation loss is weighted differently from training loss, their gap also reflects that weighting.

this screenshot shows earlier run extending to roughly 67 epochs. The revised run’s curve, history and metrics are saved in `results/high_flow/mlp/`; corresponding CNN outputs are in `results/high_flow/cnn/`. 


### Figure 5. 2D-CNN training and validation loss

![Louis’s supplied 2D-CNN training and validation loss.](images/louis_cnn_loss.png)

**Source:** 2D-CNN loss-curve screenshot. Training and validation losses decline rapidly and then flatten, with validation loss above training loss after the initial epochs. The displayed curve does not show a sustained late increase in validation loss, but it cannot by itself establish test performance or rule out overfitting.

**Run distinction:** this screenshot extends to roughly 41 epochs and documents Louis’s earlier run. Its revised curve, training history and metrics are saved in `results/high_flow/cnn/`. Keep the supplied figures distinct from the revised experiment’s numerical results.
